# Chapter 20 - Reinforcement learning

*This notebook contains all the sample code in Chapter 20.*

## Outline

- [Introduction](#Introduction)
- [Agent-environment interaction](#Interaction)
- [Tabular control algorithms](#Tabular)
    - [SARSA](#SARSA)
    - [Q-learning](#Qlearning)
- [A NumPy implementation](#Implementation)

## Introduction <a id="Introduction"></a>

**Reinforcement learning** (RL) studies how an *agent* learns to make sequential decisions by interacting with an *environment*. At each time step, the agent observes the current state, selects an action, and receives a numerical reward together with a new state. Unlike supervised learning, the agent is not told which action is correct; it must discover effective behavior through experience.

The goal is not generally to maximize the immediate reward, but the cumulative reward obtained over time. This makes RL suitable for problems such as game playing, robot control, resource allocation, and navigation, where an action may influence rewards received much later.

> An RL agent learns a policy that maps states to actions. The policy is improved from trial-and-error interactions so as to maximize the expected cumulative reward.

## Agent-environment interaction <a id="Interaction"></a>

At time step $t$, the agent observes a state\index{State} $S_t\in\mathcal{S}$ and selects an *action* $A_t\in\mathcal{A}(S_t)$. The environment then produces a **reward** $R_{t+1}$ and a new state $S_{t+1}$. A *policy* specifies how actions are selected. A stochastic policy is denoted by:
$$
\pi(a\mid s)=\Pr(A_t=a\mid S_t=s),
$$
whereas a deterministic policy directly assigns one action to each state.

An interaction may be divided into **episodes**, each ending in a terminal state, or it may continue indefinitely. Maze navigation and game playing are usually episodic, while the control of a continuously operating system is a continuing task.

The **discounted return** from time $t$ is:
$$
G_t = R_{t+1}+\gamma R_{t+2}+\gamma^2R_{t+3}+\cdots = \sum_{k=0}^{\infty}\gamma^kR_{t+k+1},
$$
where $0\leq\gamma\leq 1$ is the *discount factor*. A small value of $\gamma$ emphasizes immediate rewards, whereas a value close to one gives greater importance to long-term consequences. For continuing tasks, $\gamma<1$ also ensures that the return remains finite when rewards are bounded.

The **state-value function** of a policy $\pi$ is the expected return obtained when the agent starts from state $s$ and subsequently follows $\pi$:
$$
V^{\pi}(s) = \mathbb{E}_{\pi}\!\left[G_t\mid S_t=s\right].
$$
The **action-value function** additionally conditions on the first action:
$$
Q^{\pi}(s,a) = \mathbb{E}_{\pi}\!\left[G_t\mid S_t=s,A_t=a\right].
$$

## Tabular control algorithms <a id="Tabular"></a>

Policy evaluation estimates the value of a fixed policy. **Control** methods simultaneously improve the policy and estimate its action values. Two fundamental model-free algorithms are SARSA and Q-learning.


### SARSA <a id="SARSA"></a>

**SARSA** is an *on-policy* method: it learns the action values of the policy that is actually used to generate the data. After observing the transition $(S_t,A_t,R_{t+1},S_{t+1})$, the policy selects the next action $A_{t+1}$ and performs the update:
$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha \left[R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t) \right].
$$
Its name is derived from the sequence $(S_t,A_t,R_{t+1},S_{t+1},A_{t+1})$. Because the update includes the exploratory next action, SARSA accounts for the risks induced by its current behaviour policy.


### Q-learning <a id="Qlearning"></a>

**Q-learning** is an *off-policy* method. It may behave according to an exploratory policy, but its update uses the greedy action at the next state:
$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha \left[ R_{t+1} + \gamma\max_{a'}Q(S_{t+1},a') - Q(S_t,A_t) \right].
$$
Thus, Q-learning estimates the value of the greedy target policy while data may be generated by an $\epsilon$-greedy behaviour policy.

SARSA and Q-learning differ only in their targets, but the resulting policies may differ during training. For example, near a dangerous region, SARSA may prefer a safer route because it accounts for occasional exploratory actions, whereas Q-learning evaluates the greedy continuation.

## A NumPy implementation <a id="Implementation"></a>

The following example implements a small deterministic grid world without relying on an RL library. The agent receives $-1$ at every non-terminal step and $+10$ when it reaches the goal. An obstacle cannot be entered, and actions that would leave the grid keep the agent in its current state.

In [1]:
import numpy as np

class GridWorld:
    """Small deterministic grid-world environment."""

    ACTIONS = np.array([
        [-1, 0],   # up
        [0, 1],    # right
        [1, 0],    # down
        [0, -1],   # left
    ])

    def __init__(
        self,
        rows=4,
        cols=4,
        start=(3, 0),
        goal=(0, 3),
        obstacles=((1, 1),),
    ):
        self.rows = rows
        self.cols = cols
        self.start = tuple(start)
        self.goal = tuple(goal)
        self.obstacles = {tuple(x) for x in obstacles}

        if self.start == self.goal:
            raise ValueError("start and goal must differ")
        if self.start in self.obstacles or self.goal in self.obstacles:
            raise ValueError("start and goal cannot be obstacles")

        self.n_states = rows * cols
        self.n_actions = len(self.ACTIONS)
        self.state = self.start

    def _index(self, position):
        row, col = position
        return row * self.cols + col

    def _position(self, state):
        return divmod(int(state), self.cols)

    def reset(self):
        self.state = self.start
        return self._index(self.state)

    def step(self, action):
        if not 0 <= action < self.n_actions:
            raise ValueError("invalid action")

        candidate = tuple(np.asarray(self.state) + self.ACTIONS[action])
        row, col = candidate

        valid = (
            0 <= row < self.rows
            and 0 <= col < self.cols
            and candidate not in self.obstacles
        )
        if valid:
            self.state = candidate

        done = self.state == self.goal
        reward = 10.0 if done else -1.0
        return self._index(self.state), reward, done

An $\epsilon$-greedy action selector and a common training function for **SARSA** and **Q-learning** are given below. The code randomly resolves ties among greedy actions.

In [2]:
def epsilon_greedy(Q, state, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))

    values = Q[state]
    best_actions = np.flatnonzero(np.isclose(values, np.max(values)))
		
    return int(rng.choice(best_actions))


def train_control(
    env,
    method="q_learning",
    episodes=1000,
    alpha=0.2,
    gamma=0.95,
    epsilon=0.3,
    epsilon_min=0.02,
    epsilon_decay=0.995,
    max_steps=200,
    seed=1,
):
    """Train a tabular SARSA or Q-learning agent."""

    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    episode_returns = np.zeros(episodes)

    for episode in range(episodes):
        state = env.reset()
        action = epsilon_greedy(Q, state, epsilon, rng)

        for _ in range(max_steps):
            next_state, reward, done = env.step(action)
            episode_returns[episode] += reward

            if done:
                target = reward
            elif method == "sarsa":
                next_action = epsilon_greedy(Q, next_state, epsilon, rng)
                target = reward + gamma * Q[next_state, next_action]
            else:
                target = reward + gamma * np.max(Q[next_state])

            Q[state, action] += alpha * (target - Q[state, action])

            if done:
                break

            state = next_state
            if method == "sarsa":
                action = next_action
            else:
                action = epsilon_greedy(Q, state, epsilon, rng)

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    return Q, episode_returns


def greedy_path(env, Q, max_steps=50):
    """Return positions visited by the greedy policy."""

    state = env.reset()
    path = [env._position(state)]

    for _ in range(max_steps):
        action = int(np.argmax(Q[state]))
        state, _, done = env.step(action)
        path.append(env._position(state))
        if done:
            return path

    raise RuntimeError("the greedy policy did not reach the goal")

The two algorithms can now be compared on the same environment:

In [3]:
env = GridWorld()

Q_sarsa, returns_sarsa = train_control(env, method="sarsa")

Q_q, returns_q = train_control(env, method="q_learning")

print("SARSA path:", greedy_path(env, Q_sarsa))
print("Q-learning path:", greedy_path(env, Q_q))

print("Mean final SARSA return:", np.mean(returns_sarsa[-100:]))
print("Mean final Q-learning return:", np.mean(returns_q[-100:]))

SARSA path: [(3, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3), (0, 3)]
Q-learning path: [(3, 0), (2, 0), (2, 1), (2, 2), (1, 2), (0, 2), (0, 3)]
Mean final SARSA return: 4.92
Mean final Q-learning return: 4.99
